In [2]:
import pandas as pd
import numpy as np

In [3]:
train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")

In [4]:
train.head()

,datetime,season,holiday,workingday,weather,temp,atemp,humidity,windspeed,casual,registered,count
0,2011-01-01 00:00:00,1,0,0,1,9.84,14.395,81,0.0,3,13,16
1,2011-01-01 01:00:00,1,0,0,1,9.02,13.635,80,0.0,8,32,40
2,2011-01-01 02:00:00,1,0,0,1,9.02,13.635,80,0.0,5,27,32
3,2011-01-01 03:00:00,1,0,0,1,9.84,14.395,75,0.0,3,10,13
4,2011-01-01 04:00:00,1,0,0,1,9.84,14.395,75,0.0,0,1,1


In [5]:
test.head()

,datetime,season,holiday,workingday,weather,temp,atemp,humidity,windspeed
0,2011-01-20 00:00:00,1,0,1,1,10.66,11.365,56,26.0027
1,2011-01-20 01:00:00,1,0,1,1,10.66,13.635,56,0.0000
2,2011-01-20 02:00:00,1,0,1,1,10.66,13.635,56,0.0000
3,2011-01-20 03:00:00,1,0,1,1,10.66,12.880,56,11.0014
4,2011-01-20 04:00:00,1,0,1,1,10.66,12.880,56,11.0014


In [6]:
train.dtypes

datetime          str
season          int64
holiday         int64
workingday      int64
weather         int64
temp          float64
atemp         float64
humidity        int64
windspeed     float64
casual          int64
registered      int64
count           int64
dtype: object

In [7]:
train["datetime"] = pd.to_datetime(train["datetime"])
test["datetime"] = pd.to_datetime(test["datetime"])

In [8]:
train.dtypes

datetime      datetime64[us]
season                 int64
holiday                int64
workingday             int64
weather                int64
temp                 float64
atemp                float64
humidity               int64
windspeed            float64
casual                 int64
registered             int64
count                  int64
dtype: object

In [9]:
for df in [train, test]:
    df["year"] = df["datetime"].dt.year
    df["month"] = df["datetime"].dt.month
    df["day"] = df["datetime"].dt.day
    df["hour"] = df["datetime"].dt.hour
    df["day_of_week"] = df["datetime"].dt.dayofweek

In [10]:
train[
    ["datetime", "year", "month", "day", "hour", "day_of_week"]
].head(10)

,datetime,year,month,day,hour,day_of_week
0,2011-01-01 00:00:00,2011,1,1,0,5
1,2011-01-01 01:00:00,2011,1,1,1,5
2,2011-01-01 02:00:00,2011,1,1,2,5
3,2011-01-01 03:00:00,2011,1,1,3,5
4,2011-01-01 04:00:00,2011,1,1,4,5
5,2011-01-01 05:00:00,2011,1,1,5,5
6,2011-01-01 06:00:00,2011,1,1,6,5
7,2011-01-01 07:00:00,2011,1,1,7,5
8,2011-01-01 08:00:00,2011,1,1,8,5
9,2011-01-01 09:00:00,2011,1,1,9,5


In [11]:
for df in [train, test]:
    df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)

In [12]:
train[
    ["day_of_week", "is_weekend"]
].head(10)

,day_of_week,is_weekend
0,5,1
1,5,1
2,5,1
3,5,1
4,5,1
5,5,1
6,5,1
7,5,1
8,5,1
9,5,1


In [13]:
def add_peak_hour(df):
    weekday_peak = (
        (df["day_of_week"] < 5) &
        (df["hour"].isin([7, 8, 9, 17, 18, 19]))
    )

    weekend_peak = (
        (df["day_of_week"] >= 5) &
        (df["hour"].between(10, 18))
    )

    df["peak_hour"] = (weekday_peak | weekend_peak).astype(int)

In [14]:
add_peak_hour(train)
add_peak_hour(test)

In [15]:
train[
    ["datetime", "hour", "day_of_week", "is_weekend", "peak_hour"]
].head(20)

,datetime,hour,day_of_week,is_weekend,peak_hour
0,2011-01-01 00:00:00,0,5,1,0
1,2011-01-01 01:00:00,1,5,1,0
2,2011-01-01 02:00:00,2,5,1,0
3,2011-01-01 03:00:00,3,5,1,0
4,2011-01-01 04:00:00,4,5,1,0
5,2011-01-01 05:00:00,5,5,1,0
6,2011-01-01 06:00:00,6,5,1,0
7,2011-01-01 07:00:00,7,5,1,0
8,2011-01-01 08:00:00,8,5,1,0
9,2011-01-01 09:00:00,9,5,1,0


In [16]:
X = train.drop(columns=["count"])
y = train["count"]

In [17]:
print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (10886, 18)
y shape: (10886,)


In [18]:
X = X.drop(columns=["casual", "registered"])

In [19]:
X = X.drop(columns=["datetime"])

In [20]:
test_features = test.drop(columns=["datetime"])

In [21]:
X.columns

Index(['season', 'holiday', 'workingday', 'weather', 'temp', 'atemp',
       'humidity', 'windspeed', 'year', 'month', 'day', 'hour', 'day_of_week',
       'is_weekend', 'peak_hour'],
      dtype='str')

In [22]:
print("Number of features:", X.shape[1])

Number of features: 15


In [23]:
X = pd.get_dummies(
    X,
    columns=["season", "weather"],
    drop_first=False
)

test_features = pd.get_dummies(
    test_features,
    columns=["season", "weather"],
    drop_first=False
)

In [24]:
X, test_features = X.align(
    test_features,
    join="left",
    axis=1,
    fill_value=0
)

In [25]:
print("Training features:", X.shape)
print("Test features:", test_features.shape)

Training features: (10886, 21)
Test features: (6493, 21)


In [26]:
print("Missing values in X:")
print(X.isnull().sum().sum())

print("Missing values in test:")
print(test_features.isnull().sum().sum())

Missing values in X:
0
Missing values in test:
0


In [27]:
X.columns.tolist()

['holiday',
 'workingday',
 'temp',
 'atemp',
 'humidity',
 'windspeed',
 'year',
 'month',
 'day',
 'hour',
 'day_of_week',
 'is_weekend',
 'peak_hour',
 'season_1',
 'season_2',
 'season_3',
 'season_4',
 'weather_1',
 'weather_2',
 'weather_3',
 'weather_4']

In [28]:
from sklearn.model_selection import train_test_split

In [29]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

In [30]:
print("X_train:", X_train.shape)
print("X_valid:", X_valid.shape)
print("y_train:", y_train.shape)
print("y_valid:", y_valid.shape)

X_train: (8708, 21)
X_valid: (2178, 21)
y_train: (8708,)
y_valid: (2178,)
